# DAC 2026 V7 — Boundary Surgeon

**Goal:** preserve the proven V3 global ranking, then improve only the top-5% decision boundary using (1) a hard-negative specialist, (2) conditional hard-negative encodings, and (3) a pairwise fraud-vs-legitimate ranker.

V7 is deliberately *surgical*: claims far above and far below the audit cutoff are locked. Candidate selection is based on NR@5%, fold stability, TP gain per portfolio swap, and a predeclared deployment-churn guard.


In [1]:
import gc
import json
import math
import os
import platform
import random
import subprocess
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from catboost import CatBoostClassifier, CatBoostRanker
from catboost.utils import get_gpu_device_count
from xgboost import DMatrix, XGBClassifier

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    lgb = None
    LGBMClassifier = None
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings('ignore', category=FutureWarning)


@dataclass(frozen=True)
class Config:
    seed: int = 2026
    folds: int = 5
    fast_mode: bool = os.getenv('DAC_FAST_MODE', '0') == '1'
    cat_iterations: int = 2860
    xgb_estimators: int = 2600
    lgbm_estimators: int = 2600
    logistic_max_iter: int = 700
    meta_C: float = 1.0
    te_alpha: float = 50.0
    candidate_pool_rates: tuple = (0.10, 0.125, 0.15)
    refiner_weights: tuple = (0.70, 0.85, 1.00)
    refiner_iterations: int = 900
    min_nr5_gain: float = 0.0010
    min_fold_wins: int = 4
    max_confirmation_loss: float = 0.0010
    ap_guardrail: float = 0.0030
    min_group_nonfraud: int = 100

    # V7 frozen V3 control
    v3_candidate_rate: float = 0.10
    v3_refiner_weight: float = 1.00

    # Specialist candidate region and surgical action bands.
    specialist_candidate_rate: float = 0.10
    action_bands: tuple = ((0.040, 0.065), (0.035, 0.070))  # (locked_top_rate, outer_rate)
    surgical_weights: tuple = (0.35, 0.60, 0.85, 1.00)

    # Hard-negative classifier: target 1 means legitimate inside the high-risk pool.
    hn_iterations: int = 700
    hn_legitimate_weight: float = 12.0
    hn_te_alpha: float = 12.0
    hn_inner_folds: int = 3

    # Pairwise ranker: fraud is the winner, legitimate is the loser.
    pair_iterations: int = 650
    pair_per_legitimate: int = 32
    pair_cutoff_tau: float = 0.018

    # Full-data deployment stabilization without fold-bag smoothing.
    specialist_full_seeds: tuple = (2026, 2127, 3407)

    # Stability gates. Churn is one-way swaps / K.
    max_oof_churn_rate: float = 0.12
    max_test_churn_rate: float = 0.14
    min_fold_wins_vs_v3: int = 3
    max_worst_fold_tp_loss: int = 4
    min_confirmation_tp_gain: int = 0
    min_gain_per_100_swaps: float = 0.75


CFG = Config()
if CFG.fast_mode:
    CFG = Config(
        fast_mode=True,
        cat_iterations=180,
        xgb_estimators=180,
        lgbm_estimators=180,
        logistic_max_iter=120,
        refiner_iterations=120,
        hn_iterations=120,
        pair_iterations=120,
        specialist_full_seeds=(2026,),
        pair_per_legitimate=12,
    )

IS_KAGGLE = Path('/kaggle/input').exists()
OUTPUT_DIR = Path('/kaggle/working/v7_artifacts' if IS_KAGGLE else '/tmp/dac2026_v7_artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = Path('/kaggle/working/submission_v7.csv' if IS_KAGGLE else '/tmp/submission_v7.csv')

random.seed(CFG.seed)
np.random.seed(CFG.seed)
os.environ['PYTHONHASHSEED'] = str(CFG.seed)

GPU_COUNT = int(get_gpu_device_count())
CAT_TASK = 'GPU' if GPU_COUNT else 'CPU'
# Kaggle T4x2 => CatBoost uses both devices as 0:1.
CAT_DEVICES = ':'.join(str(i) for i in range(GPU_COUNT)) if GPU_COUNT else None
XGB_DEVICE = 'cuda' if GPU_COUNT else 'cpu'

hardware = 'CPU runtime'
if GPU_COUNT:
    hardware = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, check=False,
    ).stdout.strip().replace('\n', '; ')

display(Markdown(
    f'**Runtime:** Python {platform.python_version()}  \\n'
    f'**Mode:** {"FAST smoke" if CFG.fast_mode else "FULL competition"}  \\n'
    f'**Visible GPUs:** {GPU_COUNT}; CatBoost devices `{CAT_DEVICES or "CPU"}`; XGBoost `{XGB_DEVICE}`  \\n'
    f'**Hardware:** {hardware}  \\n'
    f'**LightGBM available:** {LIGHTGBM_AVAILABLE}'
))


**Runtime:** Python 3.12.13  \n**Mode:** FULL competition  \n**Visible GPUs:** 2; CatBoost devices `0:1`; XGBoost `cuda`  \n**Hardware:** Tesla T4, 15360 MiB; Tesla T4, 15360 MiB  \n**LightGBM available:** True

## 1. Official ranking metric and label-neutral top-k selector

V3 selects models primarily by the organizer's **Normalized Recall@5%**. AP is retained as a ranking guardrail and Brier/Protection are reported after the final ranking is locked.


In [2]:
def _as_binary(y):
    values = np.asarray(y)
    if values.ndim != 1 or not np.isin(values, [0, 1]).all():
        raise ValueError('y_true must be a one-dimensional binary array')
    return values.astype(int)


def _as_score(scores):
    values = np.asarray(scores, dtype=float)
    if values.ndim != 1 or not np.isfinite(values).all():
        raise ValueError('scores must be finite one-dimensional values')
    return values


def _as_probability(probabilities):
    values = _as_score(probabilities)
    if ((values < 0) | (values > 1)).any():
        raise ValueError('probabilities must be in [0, 1]')
    return values


def top_k_mask(scores, capacity=0.05, tie_key=None):
    score = _as_score(scores)
    if not 0 <= capacity <= 1:
        raise ValueError('capacity must be in [0, 1]')
    k = int(np.floor(capacity * len(score)))
    selected = np.zeros(len(score), dtype=bool)
    if k == 0:
        return selected
    if tie_key is None:
        order = np.argsort(-score, kind='mergesort')
        if k < len(score) and score[order[k - 1]] == score[order[k]]:
            raise ValueError('label-neutral tie_key required when a tie crosses the boundary')
    else:
        keys = pd.Series(tie_key, dtype='string').fillna('__MISSING_KEY__').to_numpy(dtype=str)
        if len(keys) != len(score):
            raise ValueError('tie_key length mismatch')
        order = np.lexsort((keys, -score))
    selected[order[:k]] = True
    return selected


def normalized_recall_at_capacity(y_true, scores, capacity=0.05, tie_key=None):
    y = _as_binary(y_true)
    score = _as_score(scores)
    selected = top_k_mask(score, capacity=capacity, tie_key=tie_key)
    k = int(selected.sum())
    p = int(y.sum())
    captured = int(y[selected].sum())
    if p == 0 or k == 0:
        return np.nan
    recall = captured / p
    recall_max = min(1.0, k / p)
    return recall / recall_max


def age_groups(age):
    values = pd.to_numeric(pd.Series(age), errors='coerce').mask(lambda s: s < 0)
    bands = pd.cut(
        values, [-np.inf, 17, 39, 59, np.inf],
        labels=['0–17', '18–39', '40–59', '60+'], right=True,
    ).astype('object')
    return pd.Series(bands, index=values.index).fillna('Missing')


def protection_score(y_true, selected, gender, age, min_nonfraud=100):
    y = _as_binary(y_true)
    selected = np.asarray(selected, dtype=bool)

    def gap(values, official_levels=None):
        series = pd.Series(values, dtype='string').fillna('Missing').str.strip().replace('', 'Missing')
        levels = official_levels if official_levels is not None else sorted(series.unique())
        fprs = []
        for level in levels:
            member = series.eq(level).to_numpy()
            legitimate = member & (y == 0)
            denominator = int(legitimate.sum())
            if denominator >= min_nonfraud and str(level) != 'Missing':
                fprs.append(int((legitimate & selected).sum()) / denominator)
        return max(fprs) - min(fprs) if len(fprs) >= 2 else 0.0

    gender_gap = gap(gender)
    age_gap = gap(age_groups(age), ['0–17', '18–39', '40–59', '60+'])
    return 1.0 - max(gender_gap, age_gap), gender_gap, age_gap


def metric_row(y_true, score, tie_key, gender, age, name, probability=None, capacity=0.05):
    y = _as_binary(y_true)
    score = _as_score(score)
    selected = top_k_mask(score, capacity, tie_key=tie_key)
    nr = normalized_recall_at_capacity(y, score, capacity, tie_key=tie_key)
    ap = average_precision_score(y, score)
    protection, gender_gap, age_gap = protection_score(
        y, selected, gender, age, min_nonfraud=CFG.min_group_nonfraud
    )
    row = {
        'candidate': name,
        'NormalizedRecall@5%': float(nr),
        'AveragePrecision': float(ap),
        'fraud_captured': int(y[selected].sum()),
        'legitimate_audits': int(((y == 0) & selected).sum()),
        'ProtectionScore': float(protection),
        'Gap_gender': float(gender_gap),
        'Gap_age': float(age_gap),
    }
    if probability is not None:
        p = _as_probability(probability)
        brier = brier_score_loss(y, p)
        row['Brier'] = float(brier)
        row['CalibrationScore'] = float(1 - brier)
    return row


def fold_nr5(y_true, score, fold_id, tie_key, capacity=0.05):
    rows = []
    for fold in sorted(np.unique(fold_id)):
        idx = np.where(fold_id == fold)[0]
        rows.append({
            'fold': int(fold) + 1,
            'NormalizedRecall@5%': normalized_recall_at_capacity(
                np.asarray(y_true)[idx], np.asarray(score)[idx], capacity, np.asarray(tie_key)[idx]
            )
        })
    return pd.DataFrame(rows)

# Metric sanity check that does not fit a model.
_t_y = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])
_t_s = np.array([.99, .98, .97, .96, .95, .40, .30, .20, .10, .00])
assert top_k_mask(_t_s, 0.2, [f'c{i}' for i in range(10)]).sum() == 2
assert 0 <= normalized_recall_at_capacity(_t_y, _t_s, 0.2, [f'c{i}' for i in range(10)]) <= 1


## 2. Data discovery, schema guardrails, and semantic typing

Only the three official competition CSV roles are discoverable. Any path containing `ground_truth` is ignored before schema inspection.


In [3]:
ID_COL = 'claim_id'
TARGET_COL = 'label'
FORBIDDEN_INPUT_TOKENS = ('ground_truth',)
SEMANTIC_CATEGORICAL = {
    'kdkc', 'dati2', 'typeppk', 'jkpst', 'jnspelsep', 'cmg', 'severitylevel', 'diagprimer'
}


def forbidden_path(path):
    text = str(path).lower()
    return any(token in text for token in FORBIDDEN_INPUT_TOKENS)


def csv_schema(path):
    return pd.read_csv(path, nrows=0).columns.tolist()


def discover_named_csv(root, filename):
    return [p for p in root.rglob(filename) if not forbidden_path(p)]


def discover_competition_files():
    if not IS_KAGGLE:
        roots = [Path.cwd() / 'data', Path.cwd().parent / 'data']
    else:
        roots = [Path('/kaggle/input')]

    trains, tests, samples = [], [], []
    for root in roots:
        if not root.exists():
            continue
        trains.extend(discover_named_csv(root, 'train.csv'))
        tests.extend(discover_named_csv(root, 'test.csv'))
        samples.extend(discover_named_csv(root, 'sample_submission.csv'))

    if not trains or not tests or not samples:
        raise FileNotFoundError('Could not find train.csv, test.csv, and sample_submission.csv')

    for train_path in sorted(trains, key=lambda p: p.stat().st_size, reverse=True):
        train_cols = csv_schema(train_path)
        if ID_COL not in train_cols or TARGET_COL not in train_cols:
            continue
        expected_test = set(train_cols) - {TARGET_COL}
        for test_path in sorted(tests, key=lambda p: p.stat().st_size, reverse=True):
            if set(csv_schema(test_path)) != expected_test:
                continue
            test_ids = pd.read_csv(test_path, usecols=[ID_COL], dtype={ID_COL: 'string'})[ID_COL]
            for sample_path in samples:
                if csv_schema(sample_path) != [ID_COL, 'fraud_probability']:
                    continue
                sample_ids = pd.read_csv(sample_path, usecols=[ID_COL], dtype={ID_COL: 'string'})[ID_COL]
                if len(test_ids) == len(sample_ids) and set(test_ids) == set(sample_ids):
                    return train_path, test_path, sample_path
    raise FileNotFoundError('No schema-consistent DAC train/test/sample trio found')


def read_claim_csv(path):
    if forbidden_path(path):
        raise ValueError(f'Forbidden helper file path: {path}')
    columns = csv_schema(path)
    string_columns = [c for c in ({ID_COL} | SEMANTIC_CATEGORICAL) if c in columns]
    return pd.read_csv(path, dtype={c: 'string' for c in string_columns}, low_memory=False)


TRAIN_PATH, TEST_PATH, SAMPLE_PATH = discover_competition_files()
train_raw = read_claim_csv(TRAIN_PATH)
test_raw = read_claim_csv(TEST_PATH)
sample_submission = read_claim_csv(SAMPLE_PATH)

assert not forbidden_path(TRAIN_PATH)
assert not forbidden_path(TEST_PATH)
assert not forbidden_path(SAMPLE_PATH)
assert list(sample_submission.columns) == [ID_COL, 'fraud_probability']
assert TARGET_COL in train_raw and TARGET_COL not in test_raw
assert not train_raw[ID_COL].duplicated().any()
assert not test_raw[ID_COL].duplicated().any()

# Reorder test exactly to the organizer's sample submission.
test_raw = (
    sample_submission[[ID_COL]].assign(_submission_order=np.arange(len(sample_submission)))
    .merge(test_raw, on=ID_COL, how='left', validate='one_to_one', indicator=True)
    .sort_values('_submission_order').reset_index(drop=True)
)
if not test_raw['_merge'].eq('both').all():
    raise ValueError('Test/sample claim_id alignment failed')
test_raw = test_raw.drop(columns=['_submission_order', '_merge'])

display(Markdown(
    f'Loaded **{len(train_raw):,} train** and **{len(test_raw):,} test** claims.  \\n'
    f'Input directory: `{TRAIN_PATH.parent}`  \\n'
    '**Integrity check:** helper files containing `ground_truth` were excluded from discovery.'
))


Loaded **160,174 train** and **40,043 test** claims.  \nInput directory: `/kaggle/input/competitions/dac-its-2026`  \n**Integrity check:** helper files containing `ground_truth` were excluded from discovery.

## 3. V2 feature package retained for V3

V3 keeps the V2 representation that was already validated: semantic code strings, code concentration, burden density, fold-safe rarity, and the limited geography/facility/diagnosis interactions. The new supervised hierarchy features are added later only through fold-safe mappings.


In [4]:
def clean_claims(frame):
    out = frame.copy()
    for column in out.columns:
        if column == TARGET_COL:
            out[column] = pd.to_numeric(out[column], errors='raise')
        elif column in ({ID_COL} | SEMANTIC_CATEGORICAL) or str(out[column].dtype) in {'object', 'string'}:
            values = out[column].astype('string').str.strip()
            out[column] = values.mask(values.eq(''), '__MISSING__').fillna('__MISSING__')
        else:
            out[column] = pd.to_numeric(out[column], errors='coerce')
    return out


def entropy_and_share(block):
    values = block.to_numpy(dtype=float)
    totals = values.sum(axis=1)
    shares = np.divide(values, totals[:, None], out=np.zeros_like(values), where=totals[:, None] > 0)
    entropy = -(np.where(shares > 0, shares * np.log(shares + 1e-12), 0)).sum(axis=1)
    maximum_share = np.divide(values.max(axis=1), totals, out=np.zeros(len(values)), where=totals > 0)
    return entropy, maximum_share


def engineer_features(frame):
    out = frame.copy()
    dx = [c for c in out.columns if c.startswith('dx2_')]
    proc = [c for c in out.columns if c.startswith('proc')]
    age = pd.to_numeric(out.get('umur'), errors='coerce')
    los = pd.to_numeric(out.get('los'), errors='coerce').clip(lower=0)

    out['age_band'] = age_groups(age).astype('string')
    out['is_pediatric'] = age.le(17).fillna(False).astype(int)
    out['is_older_adult'] = age.ge(60).fillna(False).astype(int)
    out['zero_los'] = los.eq(0).fillna(False).astype(int)
    out['log1p_los'] = np.log1p(los)
    out['age_x_log_los'] = age * out['log1p_los']

    for prefix, columns in (('diagnosis', dx), ('procedure', proc)):
        block = out[columns].apply(pd.to_numeric, errors='coerce').fillna(0).clip(lower=0)
        out[f'{prefix}_total'] = block.sum(axis=1)
        out[f'{prefix}_nonzero_groups'] = block.ne(0).sum(axis=1)
        out[f'{prefix}_max_group'] = block.max(axis=1)
        entropy, maximum_share = entropy_and_share(block)
        out[f'{prefix}_entropy'] = entropy
        out[f'{prefix}_max_share'] = maximum_share

    out['total_coded_burden'] = out['diagnosis_total'] + out['procedure_total']
    out['total_nonzero_groups'] = out['diagnosis_nonzero_groups'] + out['procedure_nonzero_groups']
    out['log1p_coded_burden'] = np.log1p(out['total_coded_burden'])
    out['procedure_to_diagnosis'] = (out['procedure_total'] + 0.5) / (out['diagnosis_total'] + 0.5)
    out['burden_per_day'] = out['total_coded_burden'] / (los.fillna(0) + 1)
    out['procedure_per_day'] = out['procedure_total'] / (los.fillna(0) + 1)
    out['diagnosis_per_day'] = out['diagnosis_total'] / (los.fillna(0) + 1)

    for source in ['kdkc', 'dati2']:
        if source in out:
            out[f'{source}_prefix2'] = out[source].astype('string').str[:2]

    interactions = {
        'cmg__severity': ('cmg', 'severitylevel'),
        'service__facility': ('jnspelsep', 'typeppk'),
        'geo__facility': ('dati2', 'typeppk'),
        'kdkc__dati2': ('kdkc', 'dati2'),
        'diagnosis__cmg': ('diagprimer', 'cmg'),
        'diagnosis__service': ('diagprimer', 'jnspelsep'),
        'diagnosis__facility': ('diagprimer', 'typeppk'),
        'cmg__facility': ('cmg', 'typeppk'),
    }
    for name, (left, right) in interactions.items():
        if left in out and right in out:
            out[name] = out[left].astype('string') + '__' + out[right].astype('string')
    return out


train_clean = clean_claims(train_raw)
test_clean = clean_claims(test_raw)
train_model = engineer_features(train_clean)
test_model = engineer_features(test_clean)

y = pd.to_numeric(train_model[TARGET_COL], errors='raise').astype(int).to_numpy()
if not np.isin(y, [0, 1]).all():
    raise ValueError('Training labels must be binary')

TIE_KEYS = train_raw[ID_COL].astype('string').to_numpy(dtype=str)
TEST_TIE_KEYS = test_raw[ID_COL].astype('string').to_numpy(dtype=str)

CONSTANT_COLUMNS = [
    c for c in train_model.columns
    if c not in {ID_COL, TARGET_COL} and train_model[c].nunique(dropna=False) <= 1
]
FEATURE_COLUMNS = [
    c for c in train_model.columns
    if c not in {ID_COL, TARGET_COL, *CONSTANT_COLUMNS}
]
CATEGORICAL_COLUMNS = [
    c for c in FEATURE_COLUMNS
    if c in SEMANTIC_CATEGORICAL or str(train_model[c].dtype) in {'object', 'string'}
]
FREQUENCY_BASE_COLUMNS = [
    c for c in ['kdkc', 'dati2', 'geo__facility', 'kdkc__dati2', 'diagnosis__facility', 'diagnosis__cmg']
    if c in FEATURE_COLUMNS
]

# claim_id is never a predictive feature.
assert ID_COL not in FEATURE_COLUMNS
assert TARGET_COL not in FEATURE_COLUMNS

display(Markdown(
    f'V3 static feature package: **{len(FEATURE_COLUMNS)}** non-constant predictors; '
    f'**{len(CATEGORICAL_COLUMNS)}** categorical; training prevalence **{y.mean():.2%}**.'
))


V3 static feature package: **81** non-constant predictors; **19** categorical; training prevalence **50.07%**.

## 4. Leakage-safe grouped validation

Exact repeated cleaned feature signatures stay inside one fold. This preserves the V2 validation contract and prevents obvious duplicate/template leakage across OOF folds.


In [5]:
signature_frame = train_clean.drop(columns=[ID_COL, TARGET_COL], errors='ignore')
SIGNATURE_GROUPS = pd.util.hash_pandas_object(signature_frame, index=False).astype('uint64').astype('string')

if SIGNATURE_GROUPS.nunique() >= CFG.folds:
    splitter = StratifiedGroupKFold(CFG.folds, shuffle=True, random_state=CFG.seed)
    PRIMARY_SPLITS = list(splitter.split(train_model, y, groups=SIGNATURE_GROUPS))
    PRIMARY_VALIDATION = 'StratifiedGroupKFold by exact cleaned feature signature'
else:
    splitter = StratifiedKFold(CFG.folds, shuffle=True, random_state=CFG.seed)
    PRIMARY_SPLITS = list(splitter.split(train_model, y))
    PRIMARY_VALIDATION = 'StratifiedKFold fallback'

FOLD_ID = np.full(len(y), -1, dtype=int)
fold_rows = []
for fold, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
    FOLD_ID[valid_idx] = fold
    overlap = len(set(SIGNATURE_GROUPS.iloc[fit_idx]) & set(SIGNATURE_GROUPS.iloc[valid_idx]))
    fold_rows.append({
        'fold': fold + 1,
        'train_n': len(fit_idx),
        'valid_n': len(valid_idx),
        'valid_fraud_rate': float(y[valid_idx].mean()),
        'signature_overlap': int(overlap),
        'role': 'selection' if fold < 3 else 'confirmation',
    })

VALIDATION_TABLE = pd.DataFrame(fold_rows)
if (FOLD_ID < 0).any() or VALIDATION_TABLE['signature_overlap'].sum() != 0:
    raise RuntimeError('Grouped fold separation contract failed')
VALIDATION_TABLE.to_csv(OUTPUT_DIR / 'primary_fold_assignments_summary.csv', index=False)
pd.DataFrame({ID_COL: TIE_KEYS, 'fold': FOLD_ID}).to_csv(OUTPUT_DIR / 'primary_fold_assignments.csv', index=False)
display(VALIDATION_TABLE)


,fold,train_n,valid_n,valid_fraud_rate,signature_overlap,role
0,1,128172,32002,0.503625,0,selection
1,2,128152,32022,0.499500,0,selection
2,3,128197,31977,0.499672,0,selection
3,4,128033,32141,0.497744,0,confirmation
4,5,128142,32032,0.503091,0,confirmation


## 5. Fold-safe hierarchical target estimates

The post-V2 error analysis showed that geography, facility, case group, and diagnosis combinations remain highly informative near the audit boundary. V3 therefore adds a small supervised hierarchy layer. Every validation value is mapped from **outer-training labels only**; test values are averaged across the five outer-fold mappings.


In [6]:
TE_SPECS = {
    'dati2_typeppk_cmg': ('dati2', 'typeppk', 'cmg'),
    'dati2_diagprimer_cmg': ('dati2', 'diagprimer', 'cmg'),
    'kdkc_typeppk_cmg': ('kdkc', 'typeppk', 'cmg'),
    'dati2_typeppk_diagprimer': ('dati2', 'typeppk', 'diagprimer'),
    'diagprimer_cmg_typeppk': ('diagprimer', 'cmg', 'typeppk'),
    'dati2_jnspelsep_typeppk': ('dati2', 'jnspelsep', 'typeppk'),
}
TE_SPECS = {
    name: tuple(c for c in columns if c in train_model.columns)
    for name, columns in TE_SPECS.items()
}
TE_SPECS = {name: columns for name, columns in TE_SPECS.items() if len(columns) >= 2}


def target_encoding_map(fit_frame, fit_target, apply_frame, columns, alpha):
    columns = list(columns)
    fit_keys = pd.MultiIndex.from_frame(fit_frame[columns].astype('string'))
    apply_keys = pd.MultiIndex.from_frame(apply_frame[columns].astype('string'))
    stats = pd.DataFrame({'_y': np.asarray(fit_target, dtype=float)}, index=fit_keys)
    stats = stats.groupby(level=list(range(len(columns))))['_y'].agg(['sum', 'count'])
    prior = float(np.mean(fit_target))
    rate = (stats['sum'] + alpha * prior) / (stats['count'] + alpha)
    mapped_rate = rate.reindex(apply_keys).to_numpy(dtype=float)
    mapped_count = stats['count'].reindex(apply_keys).to_numpy(dtype=float)
    mapped_rate = np.where(np.isfinite(mapped_rate), mapped_rate, prior)
    mapped_count = np.where(np.isfinite(mapped_count), mapped_count, 0.0)
    return mapped_rate, np.log1p(mapped_count)


def build_fold_safe_target_estimates():
    oof = pd.DataFrame(index=train_model.index)
    test_fold_frames = []
    for fold, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
        test_fold = pd.DataFrame(index=test_model.index)
        for name, columns in TE_SPECS.items():
            valid_rate, valid_count = target_encoding_map(
                train_model.iloc[fit_idx], y[fit_idx], train_model.iloc[valid_idx], columns, CFG.te_alpha
            )
            test_rate, test_count = target_encoding_map(
                train_model.iloc[fit_idx], y[fit_idx], test_model, columns, CFG.te_alpha
            )
            oof.loc[valid_idx, f'te__{name}'] = valid_rate
            oof.loc[valid_idx, f'te_count__{name}'] = valid_count
            test_fold[f'te__{name}'] = test_rate
            test_fold[f'te_count__{name}'] = test_count
        test_fold_frames.append(test_fold)
    if oof.isna().any().any():
        raise RuntimeError('OOF target-estimate matrix contains missing values')
    test_mean = sum(frame for frame in test_fold_frames) / len(test_fold_frames)
    return oof.astype(float), test_mean.astype(float)


TE_OOF, TE_TEST = build_fold_safe_target_estimates()
np.savez_compressed(
    OUTPUT_DIR / 'hierarchical_target_estimates.npz',
    oof=TE_OOF.to_numpy(dtype=np.float32),
    test=TE_TEST.to_numpy(dtype=np.float32),
    columns=np.array(TE_OOF.columns, dtype=object),
)
display(Markdown(
    f'Built **{TE_OOF.shape[1]}** fold-safe hierarchical target/support features from **{len(TE_SPECS)}** compact specifications.'
))


Built **12** fold-safe hierarchical target/support features from **6** compact specifications.

## 6. Proven V2 base layer

The base roster intentionally reuses the V2 families that already produced complementary OOF rankings: CatBoost seeds 2127 and 2026, one-hot XGBoost, one-hot LightGBM, and the regularized logistic baseline. Diversity comes from model family and seed, not a large hyperparameter search.


In [7]:
def make_ohe(min_frequency=20):
    try:
        return OneHotEncoder(handle_unknown='ignore', min_frequency=min_frequency, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', min_frequency=min_frequency, sparse=True)


def catboost_ready(frame, categorical):
    out = frame.copy()
    for column in categorical:
        out[column] = out[column].astype('string').fillna('__MISSING__').astype(str)
    return out


def tabular_preprocessor(frame, features, scale_numeric=False):
    categorical = [c for c in features if c in CATEGORICAL_COLUMNS]
    numeric = [c for c in features if c not in categorical]
    numeric_steps = [('impute', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scale', StandardScaler()))
    transformer = ColumnTransformer([
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', make_ohe(20)),
        ]), categorical),
        ('numeric', Pipeline(numeric_steps), numeric),
    ], sparse_threshold=0.3)
    return transformer


def add_frequency_features(fit_frame, valid_frame, test_frame, features):
    fit = fit_frame[features].copy()
    valid = valid_frame[features].copy()
    test = test_frame[features].copy()
    added = []
    for column in FREQUENCY_BASE_COLUMNS:
        if column not in features:
            continue
        name = f'frequency__{column}'
        mapping = fit[column].astype('string').value_counts(dropna=False) / len(fit)
        fit[name] = fit[column].astype('string').map(mapping).fillna(0).astype(float)
        valid[name] = valid[column].astype('string').map(mapping).fillna(0).astype(float)
        test[name] = test[column].astype('string').map(mapping).fillna(0).astype(float)
        added.append(name)
    return fit, valid, test, list(features) + added


def xgboost_probability(model, matrix):
    return np.asarray(model.get_booster().predict(DMatrix(matrix)), dtype=float)


def lightgbm_probability(model, matrix):
    iterations = int(getattr(model, 'n_estimators_', model.n_estimators))
    return np.asarray(model.booster_.predict(matrix, num_iteration=iterations), dtype=float)


def fit_predict_base(kind, fit_frame, fit_target, valid_frame, test_frame, features, seed, params):
    order = np.random.default_rng(seed).permutation(len(fit_target))
    fit_frame = fit_frame.iloc[order].reset_index(drop=True)
    fit_target = np.asarray(fit_target)[order]
    categorical = [c for c in features if c in CATEGORICAL_COLUMNS]

    if kind == 'Logistic':
        transformer = tabular_preprocessor(fit_frame, features, scale_numeric=True)
        model = Pipeline([
            ('transform', transformer),
            ('model', LogisticRegression(
                C=params['C'], max_iter=params['max_iter'], solver='lbfgs', random_state=seed
            )),
        ])
        model.fit(fit_frame[features], fit_target)
        return (
            model.predict_proba(valid_frame[features])[:, 1],
            model.predict_proba(test_frame[features])[:, 1],
        )

    if kind == 'CatBoost':
        fit_cb = catboost_ready(fit_frame[features], categorical)
        valid_cb = catboost_ready(valid_frame[features], categorical)
        test_cb = catboost_ready(test_frame[features], categorical)
        model_params = dict(
            loss_function='Logloss', allow_writing_files=False, verbose=False,
            random_seed=seed, task_type=CAT_TASK,
            iterations=params['iterations'], learning_rate=params['learning_rate'], depth=params['depth'],
            l2_leaf_reg=params['l2_leaf_reg'], random_strength=params['random_strength'],
            bootstrap_type='Bayesian', bagging_temperature=params['bagging_temperature'],
            max_ctr_complexity=params['max_ctr_complexity'], border_count=params['border_count'],
        )
        if CAT_DEVICES:
            model_params['devices'] = CAT_DEVICES
        model = CatBoostClassifier(**model_params)
        try:
            model.fit(fit_cb, fit_target, cat_features=categorical, verbose=False)
        except Exception as error:
            message = str(error).lower()
            if CAT_TASK != 'GPU' or not any(token in message for token in ('gpu', 'cuda', 'nccl', 'memory')):
                raise
            model_params.update(devices='0', max_ctr_complexity=1)
            model = CatBoostClassifier(**model_params)
            model.fit(fit_cb, fit_target, cat_features=categorical, verbose=False)
        return model.predict_proba(valid_cb)[:, 1], model.predict_proba(test_cb)[:, 1]

    transformer = tabular_preprocessor(fit_frame, features, scale_numeric=False)
    fit_encoded = transformer.fit_transform(fit_frame[features])
    valid_encoded = transformer.transform(valid_frame[features])
    test_encoded = transformer.transform(test_frame[features])

    if kind == 'XGBoost':
        model = XGBClassifier(
            n_estimators=params['n_estimators'], learning_rate=params['learning_rate'],
            max_depth=params['max_depth'], min_child_weight=params['min_child_weight'],
            subsample=params['subsample'], colsample_bytree=params['colsample_bytree'],
            reg_alpha=params['reg_alpha'], reg_lambda=params['reg_lambda'],
            objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
            device=XGB_DEVICE, random_state=seed, n_jobs=8,
        )
        try:
            model.fit(fit_encoded, fit_target, verbose=False)
        except Exception as error:
            message = str(error).lower()
            if XGB_DEVICE != 'cuda' or not any(token in message for token in ('gpu', 'cuda', 'memory')):
                raise
            model.set_params(device='cpu')
            model.fit(fit_encoded, fit_target, verbose=False)
        return xgboost_probability(model, valid_encoded), xgboost_probability(model, test_encoded)

    if kind == 'LightGBM':
        if not LIGHTGBM_AVAILABLE:
            raise RuntimeError('LightGBM unavailable')
        model = LGBMClassifier(
            n_estimators=params['n_estimators'], learning_rate=params['learning_rate'],
            num_leaves=params['num_leaves'], max_depth=params['max_depth'],
            min_child_samples=params['min_child_samples'], subsample=params['subsample'], subsample_freq=1,
            colsample_bytree=params['colsample_bytree'], reg_alpha=params['reg_alpha'],
            reg_lambda=params['reg_lambda'], objective='binary', random_state=seed,
            metric='None', n_jobs=-1, verbosity=-1,
        )
        model.fit(fit_encoded, fit_target)
        return lightgbm_probability(model, valid_encoded), lightgbm_probability(model, test_encoded)

    raise ValueError(f'Unknown model kind: {kind}')


CAT_PARAMS = {
    'iterations': CFG.cat_iterations,
    'learning_rate': 0.030,
    'depth': 8,
    'l2_leaf_reg': 8.0,
    'random_strength': 0.35,
    'bagging_temperature': 0.5,
    'max_ctr_complexity': 2,
    'border_count': 128,
}
XGB_PARAMS = {
    'n_estimators': CFG.xgb_estimators,
    'learning_rate': 0.035,
    'max_depth': 7,
    'min_child_weight': 6,
    'subsample': 0.85,
    'colsample_bytree': 0.82,
    'reg_alpha': 0.08,
    'reg_lambda': 7.0,
}
LGBM_PARAMS = {
    'n_estimators': CFG.lgbm_estimators,
    'learning_rate': 0.025,
    'num_leaves': 48,
    'max_depth': -1,
    'min_child_samples': 60,
    'subsample': 0.85,
    'colsample_bytree': 0.82,
    'reg_alpha': 0.10,
    'reg_lambda': 7.0,
}
LOGISTIC_PARAMS = {'C': 0.5, 'max_iter': CFG.logistic_max_iter}

BASE_ROSTER = [
    ('CatBoost V3 seed 2127', 'CatBoost', 2127, CAT_PARAMS),
    ('CatBoost V3 seed 2026', 'CatBoost', 2026, CAT_PARAMS),
    ('XGBoost V3 one-hot', 'XGBoost', 2026, XGB_PARAMS),
]
if LIGHTGBM_AVAILABLE:
    BASE_ROSTER.append(('LightGBM V3 one-hot', 'LightGBM', 2026, LGBM_PARAMS))
BASE_ROSTER.append(('Logistic V3', 'Logistic', 2026, LOGISTIC_PARAMS))

BASE_RESULTS = {}
BASE_FOLD_ROWS = []


def run_base_cv(name, kind, seed, params):
    started = time.perf_counter()
    oof = np.full(len(y), np.nan, dtype=float)
    test_folds = []
    for fold, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
        fold_started = time.perf_counter()
        fit_frame, valid_frame, evaluation_frame, used_features = add_frequency_features(
            train_model.iloc[fit_idx], train_model.iloc[valid_idx], test_model, FEATURE_COLUMNS
        )
        valid_pred, test_pred = fit_predict_base(
            kind, fit_frame, y[fit_idx], valid_frame, evaluation_frame,
            used_features, seed + fold, params,
        )
        oof[valid_idx] = np.clip(valid_pred, 1e-7, 1 - 1e-7)
        test_folds.append(np.clip(test_pred, 1e-7, 1 - 1e-7))
        nr = normalized_recall_at_capacity(y[valid_idx], oof[valid_idx], 0.05, TIE_KEYS[valid_idx])
        ap = average_precision_score(y[valid_idx], oof[valid_idx])
        BASE_FOLD_ROWS.append({
            'model': name, 'fold': fold + 1, 'NR5': float(nr), 'AP': float(ap),
            'seconds': time.perf_counter() - fold_started,
        })
        print(f'{name:24s} fold {fold + 1}/{CFG.folds} | NR5={nr:.6f} | AP={ap:.6f}')
        gc.collect()
    if np.isnan(oof).any():
        raise RuntimeError(f'{name} has incomplete OOF predictions')
    test_matrix = np.vstack(test_folds)
    test_mean = test_matrix.mean(axis=0)
    result = {
        'name': name,
        'kind': kind,
        'oof': oof,
        'test': test_mean,
        'test_folds': test_matrix,
        'runtime_minutes': (time.perf_counter() - started) / 60,
    }
    BASE_RESULTS[name] = result
    safe = name.lower().replace(' ', '_').replace('/', '_')
    np.savez_compressed(OUTPUT_DIR / f'predictions_{safe}.npz', oof=oof, test=test_mean, test_folds=test_matrix)
    return result


## 7. Train the V3 base prediction matrix

This is the expensive base layer. In full mode it reproduces the evidence-backed V2 capacity rather than searching a new parameter space.


In [8]:
for model_name, model_kind, model_seed, model_params in BASE_ROSTER:
    run_base_cv(model_name, model_kind, model_seed, model_params)

BASE_FOLD_METRICS = pd.DataFrame(BASE_FOLD_ROWS)
BASE_FOLD_METRICS.to_csv(OUTPUT_DIR / 'base_fold_metrics.csv', index=False)

BASE_SUMMARY = []
for name, result in BASE_RESULTS.items():
    BASE_SUMMARY.append(metric_row(
        y, result['oof'], TIE_KEYS,
        train_clean['jkpst'], train_clean['umur'], name,
        probability=result['oof'],
    ))
BASE_SUMMARY = pd.DataFrame(BASE_SUMMARY).sort_values('NormalizedRecall@5%', ascending=False)
BASE_SUMMARY.to_csv(OUTPUT_DIR / 'base_model_summary.csv', index=False)
display(BASE_SUMMARY)


CatBoost V3 seed 2127    fold 1/5 | NR5=0.974375 | AP=0.823612
CatBoost V3 seed 2127    fold 2/5 | NR5=0.967520 | AP=0.822901
CatBoost V3 seed 2127    fold 3/5 | NR5=0.981227 | AP=0.826049
CatBoost V3 seed 2127    fold 4/5 | NR5=0.965152 | AP=0.820677
CatBoost V3 seed 2127    fold 5/5 | NR5=0.969394 | AP=0.826127
CatBoost V3 seed 2026    fold 1/5 | NR5=0.975625 | AP=0.822777
CatBoost V3 seed 2026    fold 2/5 | NR5=0.969394 | AP=0.822808
CatBoost V3 seed 2026    fold 3/5 | NR5=0.976846 | AP=0.825627
CatBoost V3 seed 2026    fold 4/5 | NR5=0.966397 | AP=0.820863
CatBoost V3 seed 2026    fold 5/5 | NR5=0.971893 | AP=0.826518
XGBoost V3 one-hot       fold 1/5 | NR5=0.952500 | AP=0.795546
XGBoost V3 one-hot       fold 2/5 | NR5=0.948157 | AP=0.796473
XGBoost V3 one-hot       fold 3/5 | NR5=0.953066 | AP=0.797094
XGBoost V3 one-hot       fold 4/5 | NR5=0.940884 | AP=0.793408
XGBoost V3 one-hot       fold 5/5 | NR5=0.956902 | AP=0.798562
LightGBM V3 one-hot      fold 1/5 | NR5=0.960625 | AP=0

,candidate,NormalizedRecall@5%,AveragePrecision,fraud_captured,legitimate_audits,ProtectionScore,Gap_gender,Gap_age,Brier,CalibrationScore
1,CatBoost V3 seed 2026,0.971903,0.823677,7783,225,0.998746,0.000004,0.001254,0.172094,0.827906
0,CatBoost V3 seed 2127,0.971279,0.823825,7778,230,0.999232,0.000183,0.000768,0.172006,0.827994
3,LightGBM V3 one-hot,0.959291,0.805046,7682,326,0.997958,0.000033,0.002042,0.179684,0.820316
2,XGBoost V3 one-hot,0.950425,0.796136,7611,397,0.997277,0.000446,0.002723,0.184113,0.815887
4,Logistic V3,0.893731,0.746359,7157,851,0.996446,0.000814,0.003554,0.200309,0.799691


## 8. Meta features: probabilities, logits, fold-local ranks, disagreement, and hierarchy

The stacker is deliberately simple. Its job is not to relearn the raw claim table; it learns when the already-validated model families disagree near the high-risk region.


In [9]:
def probability_logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fold_local_percentile(values, fold_id):
    values = np.asarray(values, dtype=float)
    ranks = np.zeros(len(values), dtype=float)
    for fold in sorted(np.unique(fold_id)):
        idx = np.where(fold_id == fold)[0]
        ranks[idx] = pd.Series(values[idx]).rank(method='average', pct=True).to_numpy()
    return ranks


def global_percentile(values):
    return pd.Series(np.asarray(values, dtype=float)).rank(method='average', pct=True).to_numpy()


def build_meta_features(base_results, te_oof, te_test):
    oof = pd.DataFrame(index=np.arange(len(y)))
    test = pd.DataFrame(index=np.arange(len(test_model)))
    base_names = list(base_results)

    for i, name in enumerate(base_names):
        key = f'm{i}_{name.lower().replace(" ", "_").replace("-", "_")}'
        p_oof = base_results[name]['oof']
        p_test = base_results[name]['test']
        oof[f'{key}__prob'] = p_oof
        oof[f'{key}__logit'] = probability_logit(p_oof)
        oof[f'{key}__rank'] = fold_local_percentile(p_oof, FOLD_ID)
        test[f'{key}__prob'] = p_test
        test[f'{key}__logit'] = probability_logit(p_test)
        test[f'{key}__rank'] = global_percentile(p_test)

    prob_oof = np.column_stack([base_results[n]['oof'] for n in base_names])
    prob_test = np.column_stack([base_results[n]['test'] for n in base_names])
    oof['ensemble__mean'] = prob_oof.mean(axis=1)
    oof['ensemble__std'] = prob_oof.std(axis=1)
    oof['ensemble__min'] = prob_oof.min(axis=1)
    oof['ensemble__max'] = prob_oof.max(axis=1)
    test['ensemble__mean'] = prob_test.mean(axis=1)
    test['ensemble__std'] = prob_test.std(axis=1)
    test['ensemble__min'] = prob_test.min(axis=1)
    test['ensemble__max'] = prob_test.max(axis=1)

    primary = 'CatBoost V3 seed 2127'
    if primary in base_results:
        p0_oof = base_results[primary]['oof']
        p0_test = base_results[primary]['test']
        for i, name in enumerate(base_names):
            if name == primary:
                continue
            oof[f'disagreement__primary_m{i}'] = np.abs(p0_oof - base_results[name]['oof'])
            test[f'disagreement__primary_m{i}'] = np.abs(p0_test - base_results[name]['test'])

    for column in te_oof.columns:
        oof[f'hierarchy__{column}'] = te_oof[column].to_numpy(dtype=float)
        test[f'hierarchy__{column}'] = te_test[column].to_numpy(dtype=float)

    return oof.astype(float), test.astype(float)


META_OOF, META_TEST = build_meta_features(BASE_RESULTS, TE_OOF, TE_TEST)
if META_OOF.isna().any().any() or META_TEST.isna().any().any():
    raise RuntimeError('Meta feature matrix contains missing values')

np.savez_compressed(
    OUTPUT_DIR / 'meta_features.npz',
    oof=META_OOF.to_numpy(dtype=np.float32),
    test=META_TEST.to_numpy(dtype=np.float32),
    columns=np.array(META_OOF.columns, dtype=object),
)
display(Markdown(f'Meta layer: **{META_OOF.shape[1]}** numeric features.'))


Meta layer: **35** numeric features.

## 9. Cross-fitted rank stacker

The post-V2 analysis found that a linear stacker corrected substantially more top-5% false positives than any fixed CatBoost/LightGBM blend. V3 therefore uses a fixed L2 logistic meta learner and requires fold-stable NR@5% improvement before promotion.


In [10]:
def make_meta_model(seed):
    return Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(C=CFG.meta_C, max_iter=800, solver='lbfgs', random_state=seed)),
    ])


def crossfit_stacker(meta_oof, meta_test):
    oof = np.full(len(y), np.nan, dtype=float)
    for fold, (fit_idx, valid_idx) in enumerate(PRIMARY_SPLITS):
        model = make_meta_model(CFG.seed + 900 + fold)
        model.fit(meta_oof.iloc[fit_idx], y[fit_idx])
        oof[valid_idx] = model.predict_proba(meta_oof.iloc[valid_idx])[:, 1]
    if np.isnan(oof).any():
        raise RuntimeError('Stacker OOF incomplete')
    final_model = make_meta_model(CFG.seed + 999)
    final_model.fit(meta_oof, y)
    test = final_model.predict_proba(meta_test)[:, 1]
    return np.clip(oof, 1e-7, 1 - 1e-7), np.clip(test, 1e-7, 1 - 1e-7), final_model


STACK_OOF, STACK_TEST, STACK_MODEL = crossfit_stacker(META_OOF, META_TEST)
STACK_RANK_OOF = fold_local_percentile(STACK_OOF, FOLD_ID)
STACK_RANK_TEST = global_percentile(STACK_TEST)

ANCHOR_NAME = 'CatBoost V3 seed 2127'
if ANCHOR_NAME not in BASE_RESULTS:
    raise RuntimeError('V2 anchor model missing from base roster')
ANCHOR_OOF = BASE_RESULTS[ANCHOR_NAME]['oof']
ANCHOR_TEST = BASE_RESULTS[ANCHOR_NAME]['test']

STACK_COMPARISON = pd.DataFrame([
    metric_row(y, ANCHOR_OOF, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], 'V2/V3 CatBoost anchor', probability=ANCHOR_OOF),
    metric_row(y, STACK_OOF, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], 'Cross-fitted logistic stack', probability=STACK_OOF),
])

anchor_fold = fold_nr5(y, ANCHOR_OOF, FOLD_ID, TIE_KEYS).rename(columns={'NormalizedRecall@5%': 'anchor_NR5'})
stack_fold = fold_nr5(y, STACK_OOF, FOLD_ID, TIE_KEYS).rename(columns={'NormalizedRecall@5%': 'stack_NR5'})
STACK_FOLD_COMPARISON = anchor_fold.merge(stack_fold, on='fold')
STACK_FOLD_COMPARISON['delta'] = STACK_FOLD_COMPARISON['stack_NR5'] - STACK_FOLD_COMPARISON['anchor_NR5']
STACK_FOLD_COMPARISON['role'] = np.where(STACK_FOLD_COMPARISON['fold'] <= 3, 'selection', 'confirmation')

STACK_COMPARISON.to_csv(OUTPUT_DIR / 'stack_comparison.csv', index=False)
STACK_FOLD_COMPARISON.to_csv(OUTPUT_DIR / 'stack_fold_comparison.csv', index=False)
display(STACK_COMPARISON)
display(STACK_FOLD_COMPARISON)


,candidate,NormalizedRecall@5%,AveragePrecision,fraud_captured,legitimate_audits,ProtectionScore,Gap_gender,Gap_age,Brier,CalibrationScore
0,V2/V3 CatBoost anchor,0.971279,0.823825,7778,230,0.999232,0.000183,0.000768,0.172006,0.827994
1,Cross-fitted logistic stack,0.973027,0.825864,7792,216,0.999034,0.000254,0.000966,0.171220,0.828780


,fold,anchor_NR5,stack_NR5,delta,role
0,1,0.974375,0.976875,0.002500,selection
1,2,0.967520,0.971268,0.003748,selection
2,3,0.981227,0.976846,-0.004380,selection
3,4,0.965152,0.968264,0.003111,confirmation
4,5,0.969394,0.972517,0.003123,confirmation


## 10. Frozen V3 boundary control

The V3 champion is reproduced first: top 10% stack candidates, one cross-fitted CatBoost boundary refiner for OOF evidence, and one full-data refiner for test inference. V7 never allows later stages to modify claims outside the narrow boundary action band.


In [ ]:
REFINER_RAW_FEATURES = FEATURE_COLUMNS
REFINER_CATEGORICAL = [c for c in REFINER_RAW_FEATURES if c in CATEGORICAL_COLUMNS]
REFINER_TRAIN = pd.concat([
    train_model[REFINER_RAW_FEATURES].reset_index(drop=True),
    META_OOF.add_prefix('meta__').reset_index(drop=True),
], axis=1)
REFINER_TEST = pd.concat([
    test_model[REFINER_RAW_FEATURES].reset_index(drop=True),
    META_TEST.add_prefix('meta__').reset_index(drop=True),
], axis=1)


def refiner_ready(frame):
    out = frame.copy()
    for column in REFINER_CATEGORICAL:
        if column in out.columns:
            out[column] = out[column].astype('string').fillna('__MISSING__').astype(str)
    return out


def fit_refiner_model(frame, target, seed):
    params = dict(
        loss_function='Logloss', allow_writing_files=False, verbose=False,
        random_seed=seed, task_type=CAT_TASK,
        iterations=CFG.refiner_iterations, learning_rate=0.035, depth=7,
        l2_leaf_reg=10.0, random_strength=0.35, bootstrap_type='Bayesian',
        bagging_temperature=0.5, max_ctr_complexity=2, border_count=128,
    )
    if CAT_DEVICES:
        params['devices'] = CAT_DEVICES
    model = CatBoostClassifier(**params)
    ready = refiner_ready(frame)
    try:
        model.fit(ready, target, cat_features=[c for c in REFINER_CATEGORICAL if c in ready.columns], verbose=False)
    except Exception as error:
        message = str(error).lower()
        if CAT_TASK != 'GPU' or not any(token in message for token in ('gpu', 'cuda', 'nccl', 'memory')):
            raise
        params.update(devices='0', max_ctr_complexity=1)
        model = CatBoostClassifier(**params)
        model.fit(ready, target, cat_features=[c for c in REFINER_CATEGORICAL if c in ready.columns], verbose=False)
    return model


def candidate_mask_from_rank(rank_values, rate):
    return np.asarray(rank_values, dtype=float) >= (1.0 - rate - 1e-12)


def boundary_rank_score(stack_rank, refiner_probability, candidate_mask, rate, weight):
    stack_rank = np.asarray(stack_rank, dtype=float)
    refiner_probability = np.asarray(refiner_probability, dtype=float)
    candidate_mask = np.asarray(candidate_mask, dtype=bool)
    result = stack_rank.copy()
    if candidate_mask.sum() == 0:
        return result
    local_stack = np.clip((stack_rank[candidate_mask] - (1 - rate)) / rate, 0, 1)
    ref_rank = pd.Series(refiner_probability[candidate_mask]).rank(method='average', pct=True).to_numpy()
    mixed = (1 - weight) * local_stack + weight * ref_rank
    result[candidate_mask] = (1 - rate) + rate * mixed
    return np.clip(result, 0, 1)


def crossfit_v3_boundary(rate):
    candidate_all = candidate_mask_from_rank(STACK_RANK_OOF, rate)
    refiner_oof = np.full(len(y), np.nan, dtype=float)

    for fold, (_, valid_idx) in enumerate(PRIMARY_SPLITS):
        fit_idx = np.where(FOLD_ID != fold)[0]
        fit_candidates = fit_idx[candidate_all[fit_idx]]
        valid_candidates = valid_idx[candidate_all[valid_idx]]
        model = fit_refiner_model(
            REFINER_TRAIN.iloc[fit_candidates], y[fit_candidates],
            CFG.seed + 3000 + fold + int(rate * 1000)
        )
        refiner_oof[valid_candidates] = model.predict_proba(
            refiner_ready(REFINER_TRAIN.iloc[valid_candidates])
        )[:, 1]
        gc.collect()

    if np.isnan(refiner_oof[candidate_all]).any():
        raise RuntimeError('V3 refiner OOF incomplete')

    test_candidates = candidate_mask_from_rank(STACK_RANK_TEST, rate)
    full_candidates = np.flatnonzero(candidate_all)
    final_model = fit_refiner_model(
        REFINER_TRAIN.iloc[full_candidates], y[full_candidates],
        CFG.seed + 3900 + int(rate * 1000)
    )
    refiner_test = np.full(len(test_model), np.nan, dtype=float)
    refiner_test[test_candidates] = final_model.predict_proba(
        refiner_ready(REFINER_TEST.iloc[np.flatnonzero(test_candidates)])
    )[:, 1]
    return candidate_all, refiner_oof, test_candidates, refiner_test


V3_CAND_OOF, V3_REF_OOF, V3_CAND_TEST, V3_REF_TEST = crossfit_v3_boundary(CFG.v3_candidate_rate)
V3_OOF_SCORE = boundary_rank_score(
    STACK_RANK_OOF, V3_REF_OOF, V3_CAND_OOF,
    CFG.v3_candidate_rate, CFG.v3_refiner_weight
)
V3_TEST_SCORE = boundary_rank_score(
    STACK_RANK_TEST, V3_REF_TEST, V3_CAND_TEST,
    CFG.v3_candidate_rate, CFG.v3_refiner_weight
)
V3_FOLD_RANK_OOF = fold_local_percentile(V3_OOF_SCORE, FOLD_ID)
V3_RANK_TEST = global_percentile(V3_TEST_SCORE)

V3_METRIC = metric_row(
    y, V3_OOF_SCORE, TIE_KEYS, train_clean['jkpst'], train_clean['umur'],
    'Frozen V3 boundary control'
)
display(pd.DataFrame([V3_METRIC]))


## 11. Conditional hard-negative evidence

Global target encoding asks “how fraud-prone is this cohort?”. V7 instead asks a residual question inside the already-high-risk V3 pool: **when V3 thinks a claim is dangerous, how often is this cohort actually a legitimate hard negative?**

These encodings are generated leakage-safely. Outer validation rows are mapped only from outer-training candidates. Training-candidate encodings are themselves inner-cross-fitted.


In [ ]:
HARD_NEGATIVE_TARGET = (1 - y).astype(np.int8)  # 1 = legitimate hard negative
HN_SPECS = {
    'dati2_typeppk_cmg': ('dati2', 'typeppk', 'cmg'),
    'dati2_diagprimer_typeppk': ('dati2', 'diagprimer', 'typeppk'),
    'kdkc_typeppk_cmg': ('kdkc', 'typeppk', 'cmg'),
    'diagprimer_cmg_typeppk': ('diagprimer', 'cmg', 'typeppk'),
    'dati2_severity_cmg': ('dati2', 'severitylevel', 'cmg'),
    'dati2_jnspelsep_typeppk': ('dati2', 'jnspelsep', 'typeppk'),
}
HN_SPECS = {
    name: tuple(c for c in cols if c in train_model.columns)
    for name, cols in HN_SPECS.items()
}
HN_SPECS = {name: cols for name, cols in HN_SPECS.items() if len(cols) >= 2}


def hn_encoding_block(fit_idx, apply_frame, alpha=CFG.hn_te_alpha):
    out = pd.DataFrame(index=np.arange(len(apply_frame)))
    for name, cols in HN_SPECS.items():
        rate, count = target_encoding_map(
            train_model.iloc[fit_idx], HARD_NEGATIVE_TARGET[fit_idx],
            apply_frame, cols, alpha
        )
        out[f'hn_rate__{name}'] = rate
        out[f'hn_support__{name}'] = count
    return out.astype(float)


def nested_hn_features(train_candidate_idx, apply_frame, seed):
    train_candidate_idx = np.asarray(train_candidate_idx, dtype=int)
    target = HARD_NEGATIVE_TARGET[train_candidate_idx]
    groups = SIGNATURE_GROUPS.iloc[train_candidate_idx].to_numpy()
    n_splits = min(CFG.hn_inner_folds, max(2, int(np.bincount(target).min())))
    inner = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    train_enc = pd.DataFrame(index=np.arange(len(train_candidate_idx)))

    for inner_fit_rel, inner_valid_rel in inner.split(
        np.zeros(len(train_candidate_idx)), target, groups=groups
    ):
        fit_ids = train_candidate_idx[inner_fit_rel]
        valid_ids = train_candidate_idx[inner_valid_rel]
        block = hn_encoding_block(fit_ids, train_model.iloc[valid_ids])
        train_enc.loc[inner_valid_rel, block.columns] = block.to_numpy()

    if train_enc.isna().any().any():
        raise RuntimeError('Nested hard-negative encodings are incomplete')
    apply_enc = hn_encoding_block(train_candidate_idx, apply_frame)
    return train_enc.astype(float), apply_enc.astype(float)


SPECIALIST_BASE_TRAIN = REFINER_TRAIN.copy().reset_index(drop=True)
SPECIALIST_BASE_TEST = REFINER_TEST.copy().reset_index(drop=True)
for frame, stack, stack_rank, v3, v3_rank in [
    (SPECIALIST_BASE_TRAIN, STACK_OOF, STACK_RANK_OOF, V3_OOF_SCORE, V3_FOLD_RANK_OOF),
    (SPECIALIST_BASE_TEST, STACK_TEST, STACK_RANK_TEST, V3_TEST_SCORE, V3_RANK_TEST),
]:
    frame['context__stack_score'] = np.asarray(stack, dtype=float)
    frame['context__stack_rank'] = np.asarray(stack_rank, dtype=float)
    frame['context__v3_score'] = np.asarray(v3, dtype=float)
    frame['context__v3_rank'] = np.asarray(v3_rank, dtype=float)

SPECIALIST_CATEGORICAL = [c for c in REFINER_CATEGORICAL if c in SPECIALIST_BASE_TRAIN.columns]


## 12. Two complementary boundary specialists

**Hard-negative classifier:** learns `P(legitimate | V3-high-risk evidence)` with extra weight on the rare legitimate class.

**Pairwise ranker:** receives explicit fraud-winner / legitimate-loser pairs. Pairs are concentrated around similar V3 ranks and receive larger weight near the 5% cutoff, aligning training with the actual ordering error we need to fix. CatBoost's `PairLogit` directly optimizes pairwise ordering and supports explicit pair weights.


In [ ]:
def specialist_ready(frame):
    out = frame.copy()
    for c in SPECIALIST_CATEGORICAL:
        if c in out.columns:
            out[c] = out[c].astype('string').fillna('__MISSING__').astype(str)
    return out


def fit_hn_classifier(X_fit, y_hn, seed):
    params = dict(
        loss_function='Logloss', allow_writing_files=False, verbose=False,
        random_seed=seed, task_type=CAT_TASK,
        iterations=CFG.hn_iterations, learning_rate=0.035, depth=7,
        l2_leaf_reg=12.0, random_strength=0.30,
        bootstrap_type='Bayesian', bagging_temperature=0.4,
        max_ctr_complexity=2, border_count=128,
        class_weights=[1.0, CFG.hn_legitimate_weight],
    )
    if CAT_DEVICES:
        params['devices'] = CAT_DEVICES
    model = CatBoostClassifier(**params)
    Xr = specialist_ready(X_fit)
    cats = [c for c in SPECIALIST_CATEGORICAL if c in Xr.columns]
    try:
        model.fit(Xr, y_hn, cat_features=cats, verbose=False)
    except Exception as error:
        message = str(error).lower()
        if CAT_TASK != 'GPU' or not any(t in message for t in ('gpu', 'cuda', 'nccl', 'memory')):
            raise
        params.update(devices='0', max_ctr_complexity=1)
        model = CatBoostClassifier(**params)
        model.fit(Xr, y_hn, cat_features=cats, verbose=False)
    return model


def make_boundary_pairs(local_y, local_v3_rank, seed):
    local_y = np.asarray(local_y, dtype=int)
    local_v3_rank = np.asarray(local_v3_rank, dtype=float)
    fraud = np.flatnonzero(local_y == 1)
    legit = np.flatnonzero(local_y == 0)
    if len(fraud) == 0 or len(legit) == 0:
        raise RuntimeError('Pairwise candidate set needs both classes')

    rng = np.random.default_rng(seed)
    fraud_rank = local_v3_rank[fraud]
    pairs, weights = [], []
    take = min(CFG.pair_per_legitimate, len(fraud))

    for neg in legit:
        # Most informative comparisons are fraud claims with similar baseline rank.
        dist = np.abs(fraud_rank - local_v3_rank[neg])
        near_n = min(max(4, int(round(take * 0.75))), len(fraud))
        if near_n < len(fraud):
            near_rel = np.argpartition(dist, near_n - 1)[:near_n]
        else:
            near_rel = np.arange(len(fraud))

        remaining = take - len(near_rel)
        chosen_rel = list(near_rel)
        if remaining > 0:
            pool = np.setdiff1d(np.arange(len(fraud)), near_rel, assume_unique=False)
            if len(pool):
                chosen_rel.extend(rng.choice(pool, size=min(remaining, len(pool)), replace=False).tolist())

        for rel in chosen_rel:
            pos = fraud[int(rel)]
            midpoint = 0.5 * (local_v3_rank[pos] + local_v3_rank[neg])
            cutoff_focus = math.exp(-((midpoint - 0.95) / CFG.pair_cutoff_tau) ** 2)
            proximity = math.exp(-abs(local_v3_rank[pos] - local_v3_rank[neg]) / 0.02)
            pairs.append((int(pos), int(neg)))
            weights.append(1.0 + 3.0 * cutoff_focus + 1.5 * proximity)

    return np.asarray(pairs, dtype=np.int64), np.asarray(weights, dtype=np.float32)


def fit_pair_ranker(X_fit, y_fit, v3_rank_fit, seed):
    pairs, pair_weights = make_boundary_pairs(y_fit, v3_rank_fit, seed)
    params = dict(
        loss_function='PairLogit', allow_writing_files=False, verbose=False,
        random_seed=seed, task_type=CAT_TASK,
        iterations=CFG.pair_iterations, learning_rate=0.035, depth=7,
        l2_leaf_reg=12.0, random_strength=0.25,
        bootstrap_type='Bayesian', bagging_temperature=0.35,
        max_ctr_complexity=2, border_count=128,
    )
    if CAT_DEVICES:
        params['devices'] = CAT_DEVICES
    model = CatBoostRanker(**params)
    Xr = specialist_ready(X_fit)
    cats = [c for c in SPECIALIST_CATEGORICAL if c in Xr.columns]
    try:
        model.fit(Xr, y_fit, cat_features=cats, pairs=pairs, pairs_weight=pair_weights, verbose=False)
    except Exception as error:
        message = str(error).lower()
        if CAT_TASK != 'GPU' or not any(t in message for t in ('gpu', 'cuda', 'nccl', 'memory')):
            raise
        params.update(devices='0', max_ctr_complexity=1)
        model = CatBoostRanker(**params)
        model.fit(Xr, y_fit, cat_features=cats, pairs=pairs, pairs_weight=pair_weights, verbose=False)
    return model, len(pairs)


## 13. Cross-fit V7 specialists

The specialist models are trained only inside the V3 top-10% candidate pool. OOF predictions remain genuinely held out. The final test specialists are retrained on **all** high-risk training candidates, preserving V3's empirically successful full-data boundary inference, then stabilized across a small set of full-data seeds rather than fold-bagged.


In [ ]:
HN_OOF_PROB = np.full(len(y), np.nan, dtype=float)
PAIR_OOF_RAW = np.full(len(y), np.nan, dtype=float)
SPECIALIST_FOLD_ROWS = []

for fold, (_, valid_idx) in enumerate(PRIMARY_SPLITS):
    fit_idx = np.where(FOLD_ID != fold)[0]
    fit_cand = fit_idx[V3_FOLD_RANK_OOF[fit_idx] >= 1.0 - CFG.specialist_candidate_rate]
    valid_cand = valid_idx[V3_FOLD_RANK_OOF[valid_idx] >= 1.0 - CFG.specialist_candidate_rate]

    tr_hn_enc, va_hn_enc = nested_hn_features(
        fit_cand, train_model.iloc[valid_cand], CFG.seed + 6100 + fold
    )
    Xtr = pd.concat([
        SPECIALIST_BASE_TRAIN.iloc[fit_cand].reset_index(drop=True),
        tr_hn_enc.reset_index(drop=True),
    ], axis=1)
    Xva = pd.concat([
        SPECIALIST_BASE_TRAIN.iloc[valid_cand].reset_index(drop=True),
        va_hn_enc.reset_index(drop=True),
    ], axis=1)

    hn = fit_hn_classifier(Xtr, HARD_NEGATIVE_TARGET[fit_cand], CFG.seed + 6200 + fold)
    HN_OOF_PROB[valid_cand] = hn.predict_proba(specialist_ready(Xva))[:, 1]

    pair, n_pairs = fit_pair_ranker(
        Xtr, y[fit_cand], V3_FOLD_RANK_OOF[fit_cand], CFG.seed + 6300 + fold
    )
    PAIR_OOF_RAW[valid_cand] = pair.predict(specialist_ready(Xva))

    SPECIALIST_FOLD_ROWS.append({
        'fold': fold + 1,
        'train_candidates': len(fit_cand),
        'valid_candidates': len(valid_cand),
        'train_legitimate': int((y[fit_cand] == 0).sum()),
        'valid_legitimate': int((y[valid_cand] == 0).sum()),
        'pair_count': int(n_pairs),
    })
    del hn, pair, Xtr, Xva, tr_hn_enc, va_hn_enc
    gc.collect()

SPECIALIST_FOLD_TABLE = pd.DataFrame(SPECIALIST_FOLD_ROWS)
display(SPECIALIST_FOLD_TABLE)

# Convert fold-specific raw pair scores to within-fold candidate ranks to remove scale mismatch.
PAIR_OOF_RISK = np.full(len(y), np.nan, dtype=float)
HN_OOF_RISK = np.full(len(y), np.nan, dtype=float)
for fold in sorted(np.unique(FOLD_ID)):
    idx = np.where((FOLD_ID == fold) & np.isfinite(PAIR_OOF_RAW))[0]
    PAIR_OOF_RISK[idx] = pd.Series(PAIR_OOF_RAW[idx]).rank(method='average', pct=True).to_numpy()
    HN_OOF_RISK[idx] = pd.Series(1.0 - HN_OOF_PROB[idx]).rank(method='average', pct=True).to_numpy()

# Full-data OOF encodings for training final specialists, and full-data maps for test.
full_cand = np.flatnonzero(V3_FOLD_RANK_OOF >= 1.0 - CFG.specialist_candidate_rate)
test_cand = np.flatnonzero(V3_RANK_TEST >= 1.0 - CFG.specialist_candidate_rate)

FULL_HN_TRAIN = pd.DataFrame(index=np.arange(len(full_cand)))
for fold in sorted(np.unique(FOLD_ID)):
    valid_rel = np.flatnonzero(FOLD_ID[full_cand] == fold)
    fit_ids = full_cand[FOLD_ID[full_cand] != fold]
    block = hn_encoding_block(fit_ids, train_model.iloc[full_cand[valid_rel]])
    FULL_HN_TRAIN.loc[valid_rel, block.columns] = block.to_numpy()
if FULL_HN_TRAIN.isna().any().any():
    raise RuntimeError('Full-data OOF hard-negative encodings incomplete')
FULL_HN_TEST = hn_encoding_block(full_cand, test_model.iloc[test_cand])

Xfull = pd.concat([
    SPECIALIST_BASE_TRAIN.iloc[full_cand].reset_index(drop=True),
    FULL_HN_TRAIN.reset_index(drop=True),
], axis=1)
Xtest = pd.concat([
    SPECIALIST_BASE_TEST.iloc[test_cand].reset_index(drop=True),
    FULL_HN_TEST.reset_index(drop=True),
], axis=1)

hn_test_components = []
pair_test_rank_components = []
FULL_SEED_ROWS = []
for seed in CFG.specialist_full_seeds:
    hn = fit_hn_classifier(Xfull, HARD_NEGATIVE_TARGET[full_cand], seed + 7100)
    hn_test_components.append(hn.predict_proba(specialist_ready(Xtest))[:, 1])

    pair, pair_count = fit_pair_ranker(Xfull, y[full_cand], V3_FOLD_RANK_OOF[full_cand], seed + 7200)
    raw = pair.predict(specialist_ready(Xtest))
    pair_test_rank_components.append(pd.Series(raw).rank(method='average', pct=True).to_numpy())
    FULL_SEED_ROWS.append({'seed': seed, 'pair_count': int(pair_count)})
    del hn, pair
    gc.collect()

HN_TEST_PROB_CAND = np.mean(np.vstack(hn_test_components), axis=0)
HN_TEST_RISK = np.full(len(test_model), np.nan, dtype=float)
PAIR_TEST_RISK = np.full(len(test_model), np.nan, dtype=float)
HN_TEST_RISK[test_cand] = pd.Series(1.0 - HN_TEST_PROB_CAND).rank(method='average', pct=True).to_numpy()
PAIR_TEST_RISK[test_cand] = np.mean(np.vstack(pair_test_rank_components), axis=0)

# Hybrid is rank-on-rank so neither specialist dominates by scale.
HYBRID_OOF_RISK = np.full(len(y), np.nan, dtype=float)
HYBRID_TEST_RISK = np.full(len(test_model), np.nan, dtype=float)
for fold in sorted(np.unique(FOLD_ID)):
    idx = np.where((FOLD_ID == fold) & np.isfinite(HN_OOF_RISK) & np.isfinite(PAIR_OOF_RISK))[0]
    HYBRID_OOF_RISK[idx] = 0.5 * HN_OOF_RISK[idx] + 0.5 * PAIR_OOF_RISK[idx]
HYBRID_TEST_RISK[test_cand] = 0.5 * HN_TEST_RISK[test_cand] + 0.5 * PAIR_TEST_RISK[test_cand]

display(pd.DataFrame(FULL_SEED_ROWS))


## 14. Surgical reranking and portfolio-efficiency metrics

Only claims between the locked top region and the outer action boundary can change order. The score outside the action band remains exactly V3. Within the band, V7 mixes V3 local order with a specialist local order, then maps the result back into the same V3 score interval.

This lets us optimize the real residual task: remove a small number of legitimate claims from the top-5% portfolio without creating V5-style mass churn.


In [ ]:
def surgical_rerank(base_score, base_rank, specialist_score, locked_top_rate, outer_rate, weight, fold_id=None):
    base_score = np.asarray(base_score, dtype=float)
    base_rank = np.asarray(base_rank, dtype=float)
    specialist_score = np.asarray(specialist_score, dtype=float)
    out = base_score.copy()

    groups = [None] if fold_id is None else sorted(np.unique(fold_id))
    for g in groups:
        group_mask = np.ones(len(out), dtype=bool) if g is None else (np.asarray(fold_id) == g)
        action = (
            group_mask
            & (base_rank >= 1.0 - outer_rate - 1e-12)
            & (base_rank < 1.0 - locked_top_rate - 1e-12)
            & np.isfinite(specialist_score)
        )
        idx = np.flatnonzero(action)
        if len(idx) < 2:
            continue
        base_local = pd.Series(base_score[idx]).rank(method='average', pct=True).to_numpy()
        spec_local = pd.Series(specialist_score[idx]).rank(method='average', pct=True).to_numpy()
        mixed = (1.0 - weight) * base_local + weight * spec_local
        lo, hi = float(base_score[idx].min()), float(base_score[idx].max())
        if hi <= lo:
            continue
        mixed_rank = pd.Series(mixed).rank(method='average', pct=True).to_numpy()
        eps = max(1e-12, (hi - lo) * 1e-8)
        out[idx] = lo + eps + (hi - lo - 2 * eps) * mixed_rank
    return out


def portfolio_delta(base_score, challenger_score, tie_key=TIE_KEYS, capacity=0.05):
    base_mask = top_k_mask(base_score, capacity, tie_key)
    new_mask = top_k_mask(challenger_score, capacity, tie_key)
    k = int(base_mask.sum())
    swap_out = base_mask & ~new_mask
    swap_in = new_mask & ~base_mask
    swaps = int(swap_out.sum())
    base_tp = int(y[base_mask].sum())
    new_tp = int(y[new_mask].sum())
    tp_gain = new_tp - base_tp
    return {
        'K': k,
        'base_tp': base_tp,
        'new_tp': new_tp,
        'tp_gain': int(tp_gain),
        'base_fp': int(((y == 0) & base_mask).sum()),
        'new_fp': int(((y == 0) & new_mask).sum()),
        'one_way_swaps': swaps,
        'portfolio_overlap': float((base_mask & new_mask).sum() / k),
        'churn_rate': float(swaps / k),
        'gain_per_100_swaps': float(100.0 * tp_gain / swaps) if swaps else 0.0,
        'swap_in_fraud_rate': float(y[swap_in].mean()) if swap_in.any() else np.nan,
        'swap_out_fraud_rate': float(y[swap_out].mean()) if swap_out.any() else np.nan,
    }


def fold_tp_deltas(base_score, challenger_score):
    rows = []
    for fold in sorted(np.unique(FOLD_ID)):
        idx = np.where(FOLD_ID == fold)[0]
        b = top_k_mask(np.asarray(base_score)[idx], 0.05, np.asarray(TIE_KEYS)[idx])
        c = top_k_mask(np.asarray(challenger_score)[idx], 0.05, np.asarray(TIE_KEYS)[idx])
        delta = int(y[idx][c].sum() - y[idx][b].sum())
        rows.append({'fold': int(fold)+1, 'tp_delta_vs_v3': delta})
    return pd.DataFrame(rows)


def unlabeled_churn(base_score, challenger_score, tie_key, capacity=0.05):
    b = top_k_mask(base_score, capacity, tie_key)
    c = top_k_mask(challenger_score, capacity, tie_key)
    k = int(b.sum())
    swaps = int((b & ~c).sum())
    return {'swaps': swaps, 'churn_rate': swaps / k, 'overlap': int((b & c).sum()) / k}


## 15. Predeclared V7 challenger matrix and robust selection

The grid is intentionally small. V7 compares three residual views (hard-negative, pairwise, hybrid), two narrow action bands, and four mixing weights. Selection prefers TP gain, confirmation-fold behavior, and gain per swap while enforcing churn and worst-fold guardrails.


In [ ]:
SPECIALIST_VIEWS = {
    'HN': (HN_OOF_RISK, HN_TEST_RISK),
    'Pairwise': (PAIR_OOF_RISK, PAIR_TEST_RISK),
    'Hybrid': (HYBRID_OOF_RISK, HYBRID_TEST_RISK),
}

V7_CANDIDATES = {
    'V3 control': {'oof': V3_OOF_SCORE, 'test': V3_TEST_SCORE, 'family': 'V3', 'band': None, 'weight': 0.0}
}
REGISTRY_ROWS = []
FOLD_DETAIL_ROWS = []

base_metric = metric_row(
    y, V3_OOF_SCORE, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], 'V3 control'
)
base_delta = portfolio_delta(V3_OOF_SCORE, V3_OOF_SCORE)
REGISTRY_ROWS.append({**base_metric, **base_delta, 'family': 'V3', 'locked_top_rate': np.nan,
                      'outer_rate': np.nan, 'weight': 0.0, 'fold_wins': 0,
                      'confirmation_tp_gain': 0, 'worst_fold_tp_delta': 0, 'eligible': True})

for family, (spec_oof, spec_test) in SPECIALIST_VIEWS.items():
    for locked_top_rate, outer_rate in CFG.action_bands:
        for weight in CFG.surgical_weights:
            name = f'{family} band={locked_top_rate:.3f}-{outer_rate:.3f} w={weight:.2f}'
            oof_score = surgical_rerank(
                V3_OOF_SCORE, V3_FOLD_RANK_OOF, spec_oof,
                locked_top_rate, outer_rate, weight, fold_id=FOLD_ID
            )
            test_score = surgical_rerank(
                V3_TEST_SCORE, V3_RANK_TEST, spec_test,
                locked_top_rate, outer_rate, weight, fold_id=None
            )
            V7_CANDIDATES[name] = {
                'oof': oof_score, 'test': test_score, 'family': family,
                'band': (locked_top_rate, outer_rate), 'weight': weight,
            }

            met = metric_row(
                y, oof_score, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], name
            )
            delta = portfolio_delta(V3_OOF_SCORE, oof_score)
            fd = fold_tp_deltas(V3_OOF_SCORE, oof_score)
            fd['candidate'] = name
            FOLD_DETAIL_ROWS.extend(fd.to_dict('records'))
            wins = int((fd.tp_delta_vs_v3 > 0).sum())
            confirmation_gain = int(fd.loc[fd.fold.isin([4,5]), 'tp_delta_vs_v3'].sum())
            worst = int(fd.tp_delta_vs_v3.min())
            ap_delta = float(met['AveragePrecision'] - base_metric['AveragePrecision'])

            eligible = bool(
                delta['tp_gain'] > 0
                and delta['churn_rate'] <= CFG.max_oof_churn_rate
                and delta['gain_per_100_swaps'] >= CFG.min_gain_per_100_swaps
                and wins >= CFG.min_fold_wins_vs_v3
                and confirmation_gain >= CFG.min_confirmation_tp_gain
                and worst >= -CFG.max_worst_fold_tp_loss
                and ap_delta >= -0.0020
            )
            REGISTRY_ROWS.append({
                **met, **delta,
                'family': family,
                'locked_top_rate': locked_top_rate,
                'outer_rate': outer_rate,
                'weight': weight,
                'AP_delta_vs_v3': ap_delta,
                'fold_wins': wins,
                'confirmation_tp_gain': confirmation_gain,
                'worst_fold_tp_delta': worst,
                'eligible': eligible,
            })

REGISTRY = pd.DataFrame(REGISTRY_ROWS)
FOLD_DETAIL = pd.DataFrame(FOLD_DETAIL_ROWS)
REGISTRY.to_csv(OUTPUT_DIR / 'v7_candidate_registry.csv', index=False)
FOLD_DETAIL.to_csv(OUTPUT_DIR / 'v7_fold_stability.csv', index=False)

display(REGISTRY.sort_values(
    ['new_tp', 'confirmation_tp_gain', 'gain_per_100_swaps', 'churn_rate'],
    ascending=[False, False, False, True]
).head(20))


## 16. Deployment-churn guard and final choice

V5 demonstrated that a better OOF score can still fail if the hidden-test top-5% portfolio changes too aggressively. Therefore the final V7 candidate must pass an unlabeled deployment-stability check: test churn versus V3 cannot exceed the predeclared cap and cannot explode far beyond its OOF churn.


In [ ]:
TEST_CHURN_ROWS = []
for name, candidate in V7_CANDIDATES.items():
    tc = unlabeled_churn(V3_TEST_SCORE, candidate['test'], TEST_TIE_KEYS, 0.05)
    TEST_CHURN_ROWS.append({'candidate': name, **tc})
TEST_CHURN = pd.DataFrame(TEST_CHURN_ROWS)
REGISTRY = REGISTRY.merge(TEST_CHURN, on='candidate', how='left', suffixes=('', '_test'))

eligible = REGISTRY[(REGISTRY['eligible']) & (REGISTRY['candidate'] != 'V3 control')].copy()
if len(eligible):
    # Candidate-specific test cap: absolute cap plus a modest allowance over OOF churn.
    eligible['deployment_cap'] = np.minimum(
        CFG.max_test_churn_rate,
        1.50 * eligible['churn_rate'] + 0.02
    )
    eligible['deployment_ok'] = eligible['churn_rate_test'] <= eligible['deployment_cap']
    eligible_ok = eligible[eligible['deployment_ok']].copy()
else:
    eligible_ok = eligible.copy()

if eligible_ok.empty:
    FINAL_CHOICE = 'V3 control'
    SELECTION_REASON = 'No V7 challenger passed both OOF robustness and deployment-churn gates.'
else:
    best = eligible_ok.sort_values(
        ['new_tp', 'confirmation_tp_gain', 'gain_per_100_swaps', 'churn_rate', 'AveragePrecision'],
        ascending=[False, False, False, True, False]
    ).iloc[0]
    FINAL_CHOICE = str(best['candidate'])
    SELECTION_REASON = 'Highest robust TP capture after fold, efficiency, and deployment-churn gates.'

REGISTRY['selected'] = REGISTRY['candidate'].eq(FINAL_CHOICE)
REGISTRY.to_csv(OUTPUT_DIR / 'v7_candidate_registry.csv', index=False)
TEST_CHURN.to_csv(OUTPUT_DIR / 'v7_test_churn.csv', index=False)

SELECTED_OOF_SCORE = np.asarray(V7_CANDIDATES[FINAL_CHOICE]['oof'], dtype=float)
SELECTED_TEST_SCORE = np.asarray(V7_CANDIDATES[FINAL_CHOICE]['test'], dtype=float)

print('FINAL_CHOICE:', FINAL_CHOICE)
print('REASON:', SELECTION_REASON)
display(REGISTRY.sort_values(['selected','new_tp','gain_per_100_swaps'], ascending=[False,False,False]).head(15))


## 17. Rank-preserving probability map and official submission

The competition metric is ranking-based, so calibration is applied only after the final order is locked. A one-dimensional monotone logistic map converts the chosen score into valid `[0,1]` `fraud_probability` values without changing the ordering.


In [ ]:
def fit_sigmoid(score, target, seed):
    model = LogisticRegression(C=1e6, max_iter=500, solver='lbfgs', random_state=seed)
    model.fit(np.asarray(score).reshape(-1, 1), np.asarray(target))
    if float(model.coef_[0, 0]) <= 0:
        raise RuntimeError('Calibration slope must be positive')
    return model

DEPLOY_CALIBRATOR = fit_sigmoid(SELECTED_OOF_SCORE, y, CFG.seed + 7999)
FINAL_TEST_PROBABILITY = DEPLOY_CALIBRATOR.predict_proba(SELECTED_TEST_SCORE.reshape(-1, 1))[:, 1]
FINAL_TEST_PROBABILITY = np.clip(FINAL_TEST_PROBABILITY, 1e-7, 1 - 1e-7)

submission = sample_submission[[ID_COL]].copy()
submission['fraud_probability'] = FINAL_TEST_PROBABILITY

if list(submission.columns) != [ID_COL, 'fraud_probability']:
    raise RuntimeError('Submission schema invalid')
if len(submission) != len(sample_submission):
    raise RuntimeError('Submission row count mismatch')
if not submission[ID_COL].astype('string').reset_index(drop=True).equals(
    sample_submission[ID_COL].astype('string').reset_index(drop=True)
):
    raise RuntimeError('Submission claim_id order mismatch')
if not np.isfinite(submission.fraud_probability).all() or not submission.fraud_probability.between(0,1).all():
    raise RuntimeError('Invalid submission probabilities')

submission.to_csv(SUBMISSION_PATH, index=False, float_format='%.12f')

np.savez_compressed(
    OUTPUT_DIR / 'v7_scores.npz',
    v3_oof_score=V3_OOF_SCORE.astype(np.float32),
    v3_test_score=V3_TEST_SCORE.astype(np.float32),
    hn_oof_risk=HN_OOF_RISK.astype(np.float32),
    hn_test_risk=HN_TEST_RISK.astype(np.float32),
    pair_oof_risk=PAIR_OOF_RISK.astype(np.float32),
    pair_test_risk=PAIR_TEST_RISK.astype(np.float32),
    hybrid_oof_risk=HYBRID_OOF_RISK.astype(np.float32),
    hybrid_test_risk=HYBRID_TEST_RISK.astype(np.float32),
    selected_oof_score=SELECTED_OOF_SCORE.astype(np.float32),
    selected_test_score=SELECTED_TEST_SCORE.astype(np.float32),
    final_test_probability=FINAL_TEST_PROBABILITY.astype(np.float32),
    fold_id=FOLD_ID.astype(np.int8),
)

selected_row = REGISTRY.loc[REGISTRY.candidate.eq(FINAL_CHOICE)].iloc[0].to_dict()
selected_record = {
    'selected_candidate': FINAL_CHOICE,
    'selection_reason': SELECTION_REASON,
    'oof_nr5': float(selected_row['NormalizedRecall@5%']),
    'oof_tp_at_5pct': int(selected_row['new_tp']),
    'oof_tp_gain_vs_v3': int(selected_row['tp_gain']),
    'oof_legitimate_audits': int(selected_row['new_fp']),
    'oof_churn_rate': float(selected_row['churn_rate']),
    'test_churn_rate_vs_v3': float(selected_row['churn_rate_test']),
    'gain_per_100_oof_swaps': float(selected_row['gain_per_100_swaps']),
    'official_data_only': True,
    'claim_id_used_as_predictive_feature': False,
    'row_position_used_as_predictive_feature': False,
}
(OUTPUT_DIR / 'selected_config.json').write_text(json.dumps(selected_record, indent=2))
(OUTPUT_DIR / 'config.json').write_text(json.dumps(asdict(CFG), indent=2, default=str))

print(json.dumps(selected_record, indent=2))
print('Submission written:', SUBMISSION_PATH)
display(submission.head())


## 18. Controlled family challengers for leaderboard learning

The reset competition allows five submissions per day. V7 writes at most one extra submission per specialist family **only if that family produced an eligible, deployment-stable candidate**. These are controlled probes, not a license for broad leaderboard search. Keep V3 as the anchor and use leaderboard evidence to learn which residual mechanism transfers.


In [ ]:
def write_family_challenger(family):
    subset = REGISTRY[
        (REGISTRY.family == family)
        & (REGISTRY.eligible)
        & (REGISTRY.candidate != FINAL_CHOICE)
    ].copy()
    if subset.empty:
        return None
    subset['deployment_cap'] = np.minimum(CFG.max_test_churn_rate, 1.50 * subset.churn_rate + 0.02)
    subset = subset[subset.churn_rate_test <= subset.deployment_cap]
    if subset.empty:
        return None
    row = subset.sort_values(
        ['new_tp','confirmation_tp_gain','gain_per_100_swaps','churn_rate'],
        ascending=[False,False,False,True]
    ).iloc[0]
    name = str(row.candidate)
    score_oof = V7_CANDIDATES[name]['oof']
    score_test = V7_CANDIDATES[name]['test']
    cal = fit_sigmoid(score_oof, y, CFG.seed + 8800 + sum(ord(c) for c in family))
    prob = np.clip(cal.predict_proba(np.asarray(score_test).reshape(-1,1))[:,1], 1e-7, 1-1e-7)
    sub = sample_submission[[ID_COL]].copy()
    sub['fraud_probability'] = prob
    path = Path('/kaggle/working' if IS_KAGGLE else '/tmp') / f'submission_v7_{family.lower()}.csv'
    sub.to_csv(path, index=False, float_format='%.12f')
    return {'family': family, 'candidate': name, 'path': str(path), 'oof_tp': int(row.new_tp),
            'oof_nr5': float(row['NormalizedRecall@5%']), 'test_churn': float(row.churn_rate_test)}

challengers = []
for fam in ['HN','Pairwise','Hybrid']:
    rec = write_family_challenger(fam)
    if rec is not None:
        challengers.append(rec)
CHALLENGERS = pd.DataFrame(challengers)
if len(CHALLENGERS):
    CHALLENGERS.to_csv(OUTPUT_DIR / 'family_challengers.csv', index=False)
display(CHALLENGERS)


## 19. Final diagnostics

A successful V7 should ideally improve V3 with **far fewer swaps than V5**. The key diagnostics are TP gain, legitimate-audit reduction, confirmation-fold behavior, gain per 100 swaps, and test top-5% churn versus V3.


In [ ]:
final_metric = metric_row(
    y, SELECTED_OOF_SCORE, TIE_KEYS, train_clean['jkpst'], train_clean['umur'], FINAL_CHOICE
)
final_delta = portfolio_delta(V3_OOF_SCORE, SELECTED_OOF_SCORE)
final_folds = fold_tp_deltas(V3_OOF_SCORE, SELECTED_OOF_SCORE)

SUMMARY = pd.DataFrame([
    {**V3_METRIC, 'role': 'V3 champion control'},
    {**final_metric, 'role': 'V7 selected'},
])
SUMMARY.to_csv(OUTPUT_DIR / 'v7_summary.csv', index=False)
final_folds.to_csv(OUTPUT_DIR / 'v7_selected_fold_deltas.csv', index=False)

display(SUMMARY)
display(pd.DataFrame([final_delta]))
display(final_folds)
print('V3 test top-5% vs V7 selected churn:', unlabeled_churn(V3_TEST_SCORE, SELECTED_TEST_SCORE, TEST_TIE_KEYS))
print('Run complete. Main file:', SUBMISSION_PATH)
